# Khai phá luật kết hợp

Trong phần này chúng ta sử dụng thuật toán Apriori để tìm ra các mối quan hệ giữa các chỉ số chất lượng nước.

Các biến liên tục được rời rạc hóa thành các biến nhị phân để áp dụng thuật toán khai phá luật kết hợp.

Các chỉ số quan trọng của luật:

- Support: tần suất xuất hiện của luật
- Confidence: độ tin cậy của luật
- Lift: mức độ mạnh của mối quan hệ

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/water_clean.csv")

df.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,7.036752,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,333.073546,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,333.073546,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


In [2]:
df_rules = pd.DataFrame()

df_rules["ph_high"] = df["ph"] > df["ph"].median()
df_rules["hardness_high"] = df["Hardness"] > df["Hardness"].median()
df_rules["solids_high"] = df["Solids"] > df["Solids"].median()
df_rules["chloramines_high"] = df["Chloramines"] > df["Chloramines"].median()
df_rules["sulfate_high"] = df["Sulfate"] > df["Sulfate"].median()
df_rules["conductivity_high"] = df["Conductivity"] > df["Conductivity"].median()
df_rules["organic_carbon_high"] = df["Organic_carbon"] > df["Organic_carbon"].median()
df_rules["trihalomethanes_high"] = df["Trihalomethanes"] > df["Trihalomethanes"].median()
df_rules["turbidity_high"] = df["Turbidity"] > df["Turbidity"].median()

df_rules.head()

,ph_high,hardness_high,solids_high,chloramines_high,sulfate_high,conductivity_high,organic_carbon_high,trihalomethanes_high,turbidity_high
0,False,True,False,True,True,True,False,True,False
1,False,False,False,False,False,True,True,False,True
2,True,True,False,True,False,False,True,False,False
3,True,True,True,True,True,False,True,True,True
4,True,False,False,False,False,False,False,False,True


In [3]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

frequent_itemsets = apriori(df_rules, min_support=0.1, use_colnames=True)

frequent_itemsets.head()

,support,itemsets
0,0.424908,frozenset({ph_high})
1,0.500000,frozenset({hardness_high})
2,0.500000,frozenset({solids_high})
3,0.500000,frozenset({chloramines_high})
4,0.380647,frozenset({sulfate_high})


## Nhận xét

Một số luật cho thấy các chỉ số ô nhiễm thường xuất hiện cùng nhau.

Ví dụ:

- Khi sulfate cao và solids cao thì khả năng nước không an toàn tăng lên.
- Một số chỉ số hóa học có mối quan hệ chặt chẽ với nhau trong việc phản ánh chất lượng nước.

Các luật này có thể giúp phát hiện sớm các nguồn nước có nguy cơ ô nhiễm.

In [4]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

rules.sort_values(by="confidence", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [5]:
rules[['antecedents','consequents','support','confidence','lift']].head(10)

,antecedents,consequents,support,confidence,lift


In [6]:
rules_filtered = rules[(rules['confidence'] > 0.6) & (rules['lift'] > 1)]

rules_filtered[['antecedents','consequents','support','confidence','lift']].head(10)

,antecedents,consequents,support,confidence,lift


In [7]:
rules_filtered.to_csv("../outputs/association_rules.csv", index=False)